# W2-A2 — Pandas for Evals (how you measure if an LLM is any good)

**Why this matters:** in applied AI, the hardest part isn't calling the model — it's knowing whether its answers are *good*. You do that with **evals**: run the model on a set of examples, compare its output to the correct answer, and score it. That scoring is a pandas job, and it looks exactly like the table below.

Scenario: you built an LLM that classifies support tickets into a category. You ran it on 12 tickets. `expected` = the correct label (gold), `predicted` = what your model said. Your job: measure how well it did — overall **and** per category (because an aggregate score can hide a weak spot).

Rules: idiomatic pandas, no manual row loops. No `sklearn` — compute the metrics yourself.

In [1]:
import pandas as pd

# --- Given: your eval results (don't change them) ---
results = pd.DataFrame({
    'ticket_id': range(1, 13),
    'category':  ['billing','billing','tech','tech','tech','account','account','billing','tech','account','billing','tech'],
    'expected':  ['refund','refund','bug','bug','howto','login','login','refund','bug','login','charge','howto'],
    'predicted': ['refund','charge','bug','howto','howto','login','reset','refund','bug','login','charge','bug'],
})
results

,ticket_id,category,expected,predicted
0,1,billing,refund,refund
1,2,billing,refund,charge
2,3,tech,bug,bug
3,4,tech,bug,howto
4,5,tech,howto,howto
5,6,account,login,login
6,7,account,login,reset
7,8,billing,refund,refund
8,9,tech,bug,bug
9,10,account,login,login


## Task 1 — Overall accuracy
Add a boolean column `correct` (was the prediction right?), then compute the **overall accuracy** (fraction correct).
<br>Hint: `expected == predicted` gives a boolean Series; what does `.mean()` of booleans give you? (You saw this trick in A2.)

In [5]:
# add `correct` column, then print overall accuracy
results['correct'] = results['expected'] == results['predicted']

overall_accuracy = results['correct'].mean()
print(f"Overall accuracy: {overall_accuracy:.2%}  ({results['correct'].sum()}/{len(results)})")


Overall accuracy: 66.67%  (8/12)


## Task 2 — Accuracy per category
Compute accuracy **grouped by `category`**. Which category is the model worst at?
<br>Hint: `groupby('category')['correct'].mean()`. This is the whole point of evals — the overall number hides which slice is failing.

In [6]:
# accuracy by category
acc_by_category = results.groupby('category')['correct'].mean().sort_values()
print(acc_by_category)
print(f"\nWorst category: {acc_by_category.idxmin()} ({acc_by_category.min():.2%})")


category
tech       0.600000
account    0.666667
billing    0.750000
Name: correct, dtype: float64

Worst category: tech (60.00%)


## Task 3 — Error analysis
Show **only the rows the model got wrong** (the failing cases). In real evals, you read these by hand to find patterns.
<br>Bonus: for the wrong rows, what `predicted` value shows up most? (a hint at the model's favourite mistake)
<br>Hint: boolean filtering on the `correct` column; `.value_counts()` for the bonus.

In [7]:
# filter to incorrect rows; bonus: value_counts of predicted on those rows
wrong = results[~results['correct']]
print("Failing cases:")
print(wrong)

print("\nBonus — what the model wrongly predicted (most common first):")
print(wrong['predicted'].value_counts())


Failing cases:
    ticket_id category expected predicted  correct
1           2  billing   refund    charge    False
3           4     tech      bug     howto    False
6           7  account    login     reset    False
11         12     tech    howto       bug    False

Bonus — what the model wrongly predicted (most common first):
predicted
charge    1
howto     1
reset     1
bug       1
Name: count, dtype: int64


## Task 4 — Explain (in your own words)
1. Your overall accuracy is one number. Why is the **per-category** breakdown (Task 2) more useful for deciding whether to ship?
2. Looking at the failing rows (Task 3), describe one pattern you notice in the model's mistakes.

> **1.** The overall accuracy (8/12 ≈ 67%) averages every category together, so a strong slice can hide a weak one. The per-category breakdown shows the model is decent on `billing` (75%) but worst on `tech` (60%). If most of my real traffic is tech tickets, that aggregate number is dangerously optimistic — a ship decision should rest on the slice that matters, not the blended average. One number tells you *how much* it fails; the breakdown tells you *where*, which is what you can actually act on.
>
> **2.** The mistakes cluster in the `tech` category around `bug` vs `howto`: ticket 4 was a real `bug` predicted as `howto`, and ticket 12 was a real `howto` predicted as `bug` — the model confuses those two labels in *both* directions. That's a systematic, fixable boundary problem (bug vs howto is genuinely fuzzy), not random noise. The other errors (`refund`→`charge`, `login`→`reset`) are scattered one-offs by comparison.